In [1]:
import numpy as np
import pandas as pd
from collections import Counter
import warnings, os, gc, re
import matplotlib.pyplot as plt
import psutil
import swifter
# import icd10
from ast import literal_eval

# from plotnine import ggplot, aes, geom_line
from plotnine import *
# import pygal as pg

%matplotlib inline

warnings.filterwarnings('ignore')
pd.options.display.float_format = '{:.4f}'.format

In [2]:
path, dirs, files = next(os.walk("chunks_3/"))
directory = 'chunks_3/'
file_count = len(files)
year_start = 2017
year_end = 2020

In [3]:
# codes in ILLNESS*_CODE that are not ICD-10 codes
not_icd = ['MCP01', 'NSD01', 'P0000', 'P0001', 'ANC01', 'ANC02', 'FP001',
          'C19CI', 'C19CIS', 'C19FRP', 'C19IP1', 'C19IP2', 'C19IP3', 'C19IP4', 'C19T1', 'C19T2', 'C19T3', 'C19X3',
          'Z0011', 'Z0012', 'Z0013', 'Z0021', 'Z0022', 'Z003', 'Z0041', 'Z0042', 'Z0050', 'Z0051', 'Z0052', 'Z0061',
          'Z0062', 'Z0071', 'Z0072', 'Z0081', 'Z0082', 'Z0091', 'Z0092', 'Z010A', 'Z010B', 'Z010C', 'Z011A', 'Z011B',
          'Z011C', 'Z011D', 'Z011E', 'Z011F', 'Z011G', 'Z011G1', 'Z011G2', 'Z011H', 'Z011H1', 'Z011H2', 'Z011I',
          'Z01201', 'Z01202', 'Z01203', 'Z01204', 'Z01205', 'Z01206', 'Z01207', 'Z01208', 'Z01209', 'Z01210', 'Z01211',
          'Z01212', 'Z01213', 'Z01214', 'Z01215', 'Z01216', 'Z01217', 'Z01218', 'Z01219', 'Z01220', 'Z01221', 'Z01222',
          'Z01223', 'Z01224', 'Z01225', 'Z01226', 'Z0131A', 'Z0131B', 'Z0132B', 'Z0141A', 'Z0141B', 'Z0141CC', 'Z0141CL',
          'Z0142BC', 'Z0142BL', 'Z0142C', 'Z0143B', 'Z0143C', 'Z015101', 'Z0151A1', 'Z0151A2', 'Z0151B1', 'Z0151B2',
          'Z0151C1', 'Z0152A1', 'Z0152A2', 'Z0153A1', 'Z0153C1', 'Z0154A1', 'Z0154B1', 'Z0156A1', 'Z0156A2', 'Z0156B1',
          'Z0156B2', 'Z0156C1', 'Z0156C2', 'Z0157A1', 'Z0157B1', 'Z0157B2', 'Z0157C1', 'Z01591', 'Z0165', 'Z0166', 
          'Z0167', 'Z0168', 'Z0169']

#icd-10 philhealth list - 2017
df_icd10 = pd.read_excel(os.path.join(directory, 'ICD10 philhealth.xlsx'))
df_icd10 = df_icd10.set_axis(['ILLNESS1_CODE','DESCRIPTION','GROUP','CASE_RATE','PROFESSIONAL_FEE', \
                              'HEALTH_CARE_INST_FEE'], axis=1, inplace=False)

#procedures philhealth list - 2015
df_procs = pd.read_excel(os.path.join(directory, 'Procedures philhealth.xlsx'))
df_procs = df_procs.set_axis(['ILLNESS1_CODE','DESCRIPTION','CASE_RATE','PROFESSIONAL_FEE', \
                              'HEALTH_CARE_INST_FEE'], axis=1, inplace=False)

In [3]:
# counts per single categorical field
def group_categ_merge_cnt(dframe, dframe_cond, col_name_grp, ren_cols, tab_name, writer1):
    df2 = dframe.groupby([col_name_grp]).size().sort_values(ascending=False).to_frame().reset_index() \
                .set_axis([ren_cols, 'Freq.'], axis=1, inplace=False)
    df3 = dframe_cond.groupby([col_name_grp]).size().sort_values(ascending=False).to_frame().reset_index() \
                .set_axis([ren_cols, 'Freq. of 0 ACTUAL_AMT'], axis=1, inplace=False)
    df4 = df2.merge(df3, how='outer', on=col_name_grp)[[col_name_grp,'Freq.','Freq. of 0 ACTUAL_AMT']].fillna(0) 
#     df4['Freq. of 0 ACTUAL_AMT'].replace(np.nan, 0)
    df4['Percent of 0 AA'] = ((df4['Freq. of 0 ACTUAL_AMT'].values/df4['Freq.'].values)*100)
    df4['Freq.'] = df4.swifter.progress_bar(False).apply(lambda x: '{:,}'.format(x['Freq.']),axis=1)
    df4['Freq. of 0 ACTUAL_AMT'] = df4.swifter.progress_bar(False).apply(lambda x: '{:,}'.format(x['Freq. of 0 ACTUAL_AMT']),axis=1)    
    df4['Percent of 0 AA'] = df4.swifter.progress_bar(False).apply(lambda x: '{:,.5f}'.format(x['Percent of 0 AA']),axis=1)
    df4.to_excel(writer1, sheet_name=tab_name, index=False)
    del[df2, df3, df4]

In [7]:
## test script
df_all = pd.read_csv('chunks_3/test/58120.csv', low_memory=False, index_col=0).head(200)
df1 = df_all[(df_all['CLAIM_STATUS'] == 'PAID') | (df_all['CLAIM_STATUS'] == 'APPROVED FOR PAYMENT')] \
                [['INSTITUTION_NAME', 'ACTUAL_AMT']].reset_index()

df1.head(10)

,YEAR,INSTITUTION_NAME,ACTUAL_AMT
0,2013,STA. ROSA COMMUNITY HOSPITAL,9786.0000
1,2013,DOÑA MARTA MEMORIAL DISTRICT HOSPITAL,5426.0000
2,2013,AGO GENERAL HOSPITAL,29084.0000
3,2013,GILBERTO O. TEODORO MEMORIAL HOSPITAL,5880.0000
4,2013,CARMEN MEDICAL CLINIC AND HOSPITAL,9170.0000
5,2013,SOUTHERN PHILIPPINES MEDICAL CENTER,28180.9900
6,2013,DELA CRUZ MATERNITY HOSPITAL,6846.0000
7,2013,LORENZO D. ZAYCO DISTRICT HOSPITAL,11229.0000
8,2013,NEGROS ORIENTAL PROVINCIAL HOSPITAL,11912.0000
9,2013,BUKIDNON PROVINCIAL MEDICAL CENTER- MALAYBALAY...,61787.5000


In [9]:
## back up trial first
df1.to_csv('chunks_3/test/test_spread.csv', index=False)

In [33]:
## test script
writer = pd.ExcelWriter('chunks_3/test/test_totals.xlsx', engine='xlsxwriter')

group_categ_merge_cnt(df1, 
                      df1[((df1['ACTUAL_AMT'] <= 1) & (df1['ACTUAL_AMT'] >= 0)) | (df1['ACTUAL_AMT'].isna())],
                      'INSTITUTION_NAME',
                      'INSTITUTION_NAME',
                      '0 ACTUAL_AMT Clms - Inst',
                      writer)

writer.save()

In [8]:
for i in range(year_start, year_end+1):
    df_all = pd.read_csv(os.path.join(directory + 'UP-' + str(i) + '.csv'), low_memory=False, index_col=0)

    df1 = df_all[(df_all['CLAIM_STATUS'] == 'PAID') | (df_all['CLAIM_STATUS'] == 'APPROVED FOR PAYMENT')] \
                [['INSTITUTION_NAME', 'ACTUAL_AMT']].reset_index()
    
    del[df_all]
#     df1.to_csv(os.path.join(directory, str(i) + '_1st_500.xlsx'), index=False)
    
    print(i)
#     print(len(df_all.index))
    
    writer = pd.ExcelWriter(os.path.join(directory, str(i) + '_totals.xlsx'), engine='xlsxwriter')
    
    group_categ_merge_cnt(df1, 
                          df1[(df1['ACTUAL_AMT'] <= 1) | (df1['ACTUAL_AMT'].isna())],
                          'INSTITUTION_NAME',
                          'INSTITUTION_NAME',
                          '0 ACTUAL_AMT Clms - Inst',
                          writer)
    
    writer.save()
    
    del[df1]    
    gc.collect()
    gc.collect()
    gc.collect()    

2017
2018
2019
2020
